# Final Project - ST 554
Author: Max Campbell

## Part 1 - Fitting a model using MLlib

In this assignment, we will demonstrate the capabilities of building models via pySpark using machine learning tools and data streaming! In particular, we want to build a pipeline that we can use to fit data to an elastic net model (chosen because we can cross-validate to find the optimal tuning parameters, which in turn will allow the model to keep itself relatively stable at high levels of complexity), and use that pipeline to predict new data against the model quickly and effectively. Let's begin by reading in the base dataset that we will use to fit the model. Our dataset of choice is power readings from Tetouan, Morocco as it relates to various environmental factors such as temperature, humidity, and time of day. The variable of interest is `Power_Zone_3`, representing power readings from a subsection of the city. In this context, we imagine that we are anticipating the tools used to measure `Power_Zone_3` are going offline soon, and we want a way to predict the power output while the tools are offline. Let's go ahead and get started!

In [1]:
#Load in necessary modules
import pandas as pd
from pyspark.sql import SparkSession

#Read in data as a pandas DataFrame
power = pd.read_csv("power_ml_data.csv")

#Initialize Spark session
spark = SparkSession.builder.master('local[*]').appName('FP') \
    .config("spark.sql.ansi.enabled", "false").getOrCreate()

#Read pandas DF into Spark
power = spark.createDataFrame(power)

power.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 15:32:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/27 15:32:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/27 15:32:57 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/27 15:32:57 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/27 15:32:57 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

Now that we've got our data in Spark, it's time to start setting up the pipeline. The transformations that we will be performing to this base dataset will also be performed on any future data that we read in, which is why using Spark/MLlib is a good tool for this job. Let's start with the `Hour` variable. We want to understand whether a power reading was taken (roughly) at night or day, so we will start with binarizing this variable to create an indicator of night-time readings vs day-time readings. Note that we may also have to convert `Hour` to a `DoubleType` first to accomplish this.

In [2]:
#Check data types for the dataframe
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [3]:
#Hour is a long, but we want double type, so we will convert it and then binarize it
#Import necessary modules
from pyspark.ml.feature import SQLTransformer, Binarizer
from pyspark.ml import Pipeline

#Convert Hour to Double, rename Power_Zone_3 to label
doubleTypeConverter = SQLTransformer(statement = "SELECT *, CAST(Hour AS DOUBLE) AS Hour_d, Power_Zone_3 AS label FROM __THIS__")

#Binarize Hour by whether the value is less than 6.5 or not
hourBinarizer = Binarizer(threshold = 6.5, inputCol = "Hour_d", outputCol = "isDaytime")

Next, we will need to one-hot encode the Month column so that the model can read the categorical data in a format that it can process efficiently.

In [4]:
#Import necessary modules
from pyspark.ml.feature import OneHotEncoder

#OHE Month
oneHotEncoder = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_ohe"])

After that, we will do a principal components analysis (PCA) on the environmental factors `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`. This will involve doing a PCA fit on these variables, which we will then use as the transformer that fits into our overall pipeline.

In [5]:
#Import necessary modules
from pyspark.ml.feature import VectorAssembler, PCA

#Assemble PCA features into a model
pcaVecAssembler = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol = "pcaFeatures")

#Fit the PCA model
pca = PCA(k = 2, inputCol = "pcaFeatures", outputCol = "pcaOutputs")

pipeline = Pipeline(stages = [doubleTypeConverter, hourBinarizer, oneHotEncoder, pcaVecAssembler, pca])
testing = pipeline.fit(power)

Now we can set up for the Elastic Net model, by assembling our desired features and defining the response variable.

In [6]:
#Define features
vecAssembler = VectorAssembler(inputCols = ["pcaFeatures", "isDaytime", "Power_Zone_1", "Power_Zone_2", "Month_ohe"], outputCol = "features")

We are ready to cross-validate the model now!

In [7]:
#Import necessary modules
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

#Define model
lr = LinearRegression()

#Define grid of parameters to cross-validate
grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
    
#Define evaluator
evaluator = RegressionEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="rmse"
)

#Define pipeline
pipeline = Pipeline(stages = [doubleTypeConverter, hourBinarizer, oneHotEncoder, pcaVecAssembler, pca, vecAssembler, lr])

#Define CV
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = grid,
                          evaluator = evaluator,
                          parallelism = 128,
                          numFolds = 5)

#Fit power data to CV
fitted_crossval = crossval.fit(power)

26/04/27 15:33:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/27 15:33:30 WARN Instrumentation: [699b7cc9] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:30 WARN Instrumentation: [6fca5550] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:30 WARN Instrumentation: [72cd797d] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:30 WARN Instrumentation: [4f2b642e] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:30 WARN Instrumentation: [fd98b319] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:31 WARN Instrumentation: [1fea7d86] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:32 WARN Instrumentation: [72ba9327] regP

In [13]:
#Obtain the best model and show the best parameter values
best_model = fitted_crossval.bestModel
best_lr_model = best_model.stages[-1] #Gets the model fit at the last stage of the pipeline

print("Regression Parameter: ", best_lr_model.getRegParam())
print("Elastic Net Parameter: ", best_lr_model.getElasticNetParam())

#Get CV error and print that as well
cv_err = sum(fitted_crossval.avgMetrics) / len(fitted_crossval.avgMetrics)
print("Average CV error: ", cv_err)

Regression Parameter:  0.5
Elastic Net Parameter:  0.05
Average CV error:  2124.963937978689


We see that our best performing model returned a regression parameter of 0.5 and an elastic net parameter of 0.05, so these are the values we will use in our model fit for predicting future data! Let's go ahead and compare this model to the whole training set now so we have an idea of what our Root Mean Squared Error (RMSE) is.

In [14]:
#Import necessary modules
import numpy as np

#Predict on the fitted model
prediction_df = fitted_crossval.transform(power)

#Calculate RMSE
rmse = evaluator.evaluate(prediction_df)
print("RMSE: ", rmse)

RMSE:  2124.154490323416


We are also interested in the residuals for these predictions. This will help us understand how close our predictions are on a case-by-case basis.

In [15]:
#Create a residual column
trimmed_df = prediction_df.withColumn("residual", prediction_df.label - prediction_df.prediction)[["label", "prediction", "residual"]]
trimmed_df.show()

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20203.987122100694| 36.97673789930559|
|20131.08434| 18012.08308052219|2119.0012594778127|
|19668.43373|17558.846315787905|2109.5874142120956|
|18899.27711|16939.637697748807| 1959.639412251192|
|18442.40964|16338.102891094724|2104.3067489052773|
|18130.12048|15859.139619515281|2270.9808604847203|
|17945.06024|15419.933509440893|2525.1267305591064|
|17459.27711|15041.889305957233| 2417.387804042766|
|17025.54217|14624.397962962601|2401.1442070373996|
|16794.21687|14285.106165766618|2509.1107042333824|
|16638.07229|14010.339151506538|2627.7331384934623|
|16395.18072|13779.185290140278| 2615.995429859722|
|16117.59036|13400.660687816198| 2716.929672183802|
| 15822.6506|12945.452316288818|2877.1982837111827|
|15672.28916|12776.870094494367|2895.4190655056336|
|15597.10843|12624.237465341179|2972.8709646588213|
|15510.36145

Now we have everything we need to begin evaluating new data!

## Part 2 - Streaming new data into a model

To begin, let's set up a stream reader that watches the `streamdata` folder in this project's directory for new CSV files.

In [26]:
#Set up a stream reader
stream = spark \
    .readStream \
    .schema(power.schema) \
    .option("header", "true") \
    .csv("streamdata")

Next, we want to do two separate things with the data we read in. The first is we want to create predictions and residuals for each new observation (and include `label`, representing `Power_Zone_3` as it did in the previous section). We can use the fitted model from the previous part as a transformer to accomplish this. The second thing we want to do is modify the `Power_Zone_3` column to read as `label`, so that we can demonstrate how performing a join on a stream works!

In [27]:
#Fit the new data and obtain the label, prediction, and residual reading

power_predict = fitted_crossval.transform(stream)[["label", "prediction"]]
power_predict = power_predict.withColumn("residual", power_predict.label - power_predict.prediction)
power_label = stream.withColumnRenamed("Power_Zone_3", "label")

#Perform an inner join on power_predict and power_label
power_joined = power_predict.join(power_label, "label", "inner")

Assuming everything went to plan, we should be good to start writing our stream to the console! Let's get it started and see.

In [34]:
#Write stream to console using append mode
query = power_joined.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/27 16:21:37 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8fd83104-310a-4c6c-9366-422c485b15af. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 16:21:37 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|19106.62614|21694.793569729023|-2588.167429729023|      21.17|    65.7|     4.924|                36.66|        26.44| 45903.54486| 29124.89627|   10|  18|
|20009.63855|17924.852898169484| 2084.785651830516|      18.58|   65.84|     0.075|                0.059|        0.037| 36209.23077|  28710.7438|   11|  20|
|29851.56923|27784.876139564487| 2066.693090435514|      21.63|    80.8|      4.92|                0.088|        0.107

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16431.34796| 20332.31984310041| -3900.97188310041|       25.5|    92.5|     4.913|                0.486|        0.374| 26952.45283| 17726.29356|    8|   6|
|25697.12563|25312.378734248818|  384.746895751181|      13.12|    75.9|     0.074|                0.081|        0.145| 43114.57627| 25754.40729|    2|  19|
|18929.87854| 16744.98059370653| 2184.897946293473|      21.99|   61.85|     0.072|                958.0|        150.9

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14428.91566|15828.228079216315| -1399.312419216314|      12.35|    77.8|     0.074|                 0.04|        0.093| 33950.76923| 28565.70248|   11|  22|
|17719.51807|12064.496764707332| 5655.0213052926665|      19.41|    72.5|      0.07|                205.7|        202.7| 29009.23077| 24385.53719|   11|  12|
|11100.55385|12830.186548717593|-1729.6326987175926|      21.24|    79.2|     0.069|                356.2|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12815.56231|12490.033009290142|  325.5293007098571|      22.21|   66.37|     4.923|                357.5|        205.7| 34289.01532| 19609.54357|   10|  13|
|    16704.0|20043.965136930437|-3339.9651369304374|      16.15|    84.3|     0.075|                65.09|        59.52| 34225.18837| 16456.61914|    4|  10|
|25283.85542| 26450.20183888832| -1166.346418888319|      15.74|    79.4|     0.078|                12.32|       

...

-------------------------------------------
Batch: 5
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16230.60729| 16890.33174196407|-659.7244519640699|      19.87|    75.6|     4.923|                369.5|        326.7|  33200.2623|  22562.2291|    5|  10|
|19246.26506| 17885.48056668655| 1360.784493313451|      15.37|    75.8|     0.087|                0.059|        0.074| 36461.53846|  30867.7686|   11|  21|
|16329.02821|21387.150969965816|-5058.122759965816|      29.04|   43.88|      4.92|                780.0|        39.69

-------------------------------------------
Batch: 6
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9542.376951| 11234.75284777227|-1692.3758967722697|      14.34|   44.56|     0.077|                367.9|        40.62| 30734.60076| 26091.43909|   12|  15|
|18411.01215|18783.615042506346| -372.6028925063474|      27.46|   48.23|     4.923|                583.2|         83.1| 34685.90164| 21826.62539|    5|  16|
|14235.01508|12769.161450282689| 1465.8536297173105|      6.706|    89.8|     0.071|                 0.07|       

-------------------------------------------
Batch: 7
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9386.794718| 12004.06478025614|-2617.2700622561406|      17.13|   69.13|     0.074|                183.7|        168.9| 31458.55513| 26146.67076|   12|  16|
|27861.06583| 25872.08794894518| 1988.9778810548196|      35.38|   14.54|     4.903|                805.0|        130.1| 39565.63818| 28925.44879|    8|  12|
|12714.36159| 8637.386982696738|  4076.974607303262|      22.56|   46.09|     4.933|                380.8|       

-------------------------------------------
Batch: 8
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9058.343337|12044.417238358743|-2986.0739013587427|      15.81|   64.96|      0.08|                127.2|        128.9| 31531.55894| 24467.62811|   12|  10|
|33344.20063|32474.019280037737|  870.1813499622622|      23.58|   64.55|     4.902|                 0.08|        0.089| 48196.04883|  31662.5132|    8|  21|
|16070.11055|17362.837068963334|-1292.7265189633345|      11.03|    75.4|     4.914|                213.0|       

-------------------------------------------
Batch: 9
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16628.36364| 16441.44597685245| 186.91766314754932|      13.57|    84.9|     4.923|                0.073|        0.159| 25358.88052| 13850.10183|    4|   4|
|13720.64516| 12219.99570419639| 1500.6494558036102|       9.73|   64.55|     0.082|                0.073|        0.082| 21557.10638| 12621.95122|    3|   3|
|16426.88458|16236.154889421185|  190.7296905788171|       20.9|    78.9|     4.916|                0.062|       

In [35]:
#Manually stop query if necessary
query.stop()

26/04/27 16:23:54 WARN DAGScheduler: Failed to cancel job group d1fb1bcb-d99d-4c5b-9edf-7cd423bbae01. Cannot find active jobs for it.
26/04/27 16:23:54 WARN DAGScheduler: Failed to cancel job group d1fb1bcb-d99d-4c5b-9edf-7cd423bbae01. Cannot find active jobs for it.


After testing the stream against sample data randomly selected from a new set of observations (and sent to our streaming destination via a python script), it appears that we are making new predictions on the data as intended!